In [ ]:
# Imports: make the local src-layout package available before importing it.
from pathlib import Path
import sys

import matplotlib.pyplot as plt

project_root = Path.cwd().resolve()
source_root = project_root / "src"
if not (project_root / "pyproject.toml").is_file() or not source_root.is_dir():
    raise RuntimeError("Open user_notebook.ipynb from the repository root.")
if str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))

from neutral_atom_mht import ClassicalSolver, HPC, HPCConfig
from cell_data import DATASET_NAME, load_tiff, raw_frame_path

In [ ]:
# Configuration: select one real frame and construct the processing objects.
dataset_root = project_root / "data" / DATASET_NAME
frame = 0
frame_path = raw_frame_path(dataset_root, frame)
if not frame_path.is_file():
    raise FileNotFoundError(
        f"Missing {frame_path}. Place the sequence-01 TIFFs under {dataset_root}."
    )

config = HPCConfig()
controller = HPC(config, sequence="01")
solver = ClassicalSolver()

In [ ]:
# Run: detect the cells, create tracks, and show the detections on the source frame.
image = load_tiff(frame_path)
prepared = controller.prepare_frame(image, frame=frame)
solver_result = controller.solve(prepared, solver)
result = controller.advance(prepared, solver_result)
detections = prepared.observed_frame.detection.detections

figure, axis = plt.subplots(figsize=(10, 8))
axis.imshow(image, cmap="gray")
axis.scatter(
    [detection.x_px for detection in detections],
    [detection.y_px for detection in detections],
    s=28,
    facecolors="none",
    edgecolors="tab:red",
    linewidths=0.9,
    label="detected cell",
)
axis.set_title(f"Sequence 01, frame {frame:03d}: {len(detections)} detections")
axis.set_axis_off()
axis.legend(loc="upper right")
plt.show()

{
    "frame_path": str(frame_path),
    "detections": len(detections),
    "initialized_tracks": len(result.tracks),
    "solver": solver.solver_name,
}